# Glyph — Финальный evaluation (этап 9/D) на Kaggle

Считает метрики для **всех 6 условий**:
- baseline (без RAG, без LoRA)
- rag_only (RAG без LoRA)
- rag_lora_r4 / r8 / r16 (LoRA на сырых текстах, E1-E3)
- **rag_lora_instruct** (новое: LoRA на synthetic Q&A — E5_instruct)

## Настройки Kaggle (справа, Settings)
- **Accelerator:** `GPU T4 x2`
- **Internet:** `ON`

После «Save Version» выполняется в фоне, вкладку можно закрыть.

## Время
- Perplexity: ~5 мин (6 моделей × 3 автора, forward pass)
- Generation: ~12 мин (6 условий × 10 вопросов × 3 автора = 180 ответов)
- **Итого ~20-25 минут на T4**.

## 1. GPU check

In [ ]:
!nvidia-smi | head -15
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Клон репозитория

In [ ]:
import os
os.chdir('/kaggle/working')
!rm -rf glyph
!git clone --depth 1 https://github.com/tearfu1/glyph.git
os.chdir('/kaggle/working/glyph/ml')

# Проверяем наличие ключевых артефактов
import os.path
for path in [
    'data/retrieval_cache.json',
    'models/adapters/dostoevsky/E3_high_rank/adapter_model.safetensors',
    'models/adapters/dostoevsky/E5_instruct/adapter_model.safetensors',
]:
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  {status}: {path}')

## 3. Зависимости

`--no-deps` на peft/accelerate — чтобы не сломать Kaggle-preinstalled torch.

In [ ]:
!pip install -q --upgrade --no-deps 'accelerate>=0.33.0'
!pip install -q --no-deps peft==0.18.1
!pip install -q sentence-transformers==2.7.0 razdel==0.5.0 pymorphy3==2.0.2 datasets==2.19.0

print('--- Проверка ---')
!pip show accelerate | grep -E '^(Name|Version)'
!pip show peft | grep -E '^(Name|Version)'

from accelerate.utils.memory import clear_device_cache
print('OK: clear_device_cache импортируется')

## 4. Запуск evaluation

Флаг `--use-cache` читает retrieval из JSON (Qdrant не нужен).

In [ ]:
!python -m scripts.run_evaluation --use-cache data/retrieval_cache.json --num-questions 10

## 5. Просмотр таблиц

In [ ]:
import pandas as pd

print('=== Perplexity ===')
ppl = pd.read_csv('models/evaluation/perplexity.csv')
# Сводная: строки = авторы, колонки = модели
ppl_pivot = ppl.pivot(index='author', columns='model', values='perplexity').round(2)
print(ppl_pivot)

print('\n=== Semantic similarity ===')
sem = pd.read_csv('models/evaluation/semantic_similarity.csv')
sem_pivot = sem.pivot(index='author', columns='condition', values='semantic_similarity').round(3)
print(sem_pivot)

print('\n=== Style metrics (краткая сводка) ===')
style = pd.read_csv('models/evaluation/style_metrics.csv')
print(style[['author', 'condition', 'ttr', 'avg_sentence_length', 'hapax_ratio']].round(3))

## 6. Примеры генерации: сравнение условий на одном вопросе

In [ ]:
import json
from collections import defaultdict

with open('models/evaluation/generation_samples.jsonl') as f:
    samples = [json.loads(l) for l in f]

by_aq = defaultdict(list)
for s in samples:
    by_aq[(s['author'], s['question'])].append(s)

CONDITION_ORDER = ['baseline', 'rag_only', 'rag_lora_r4', 'rag_lora_r8', 'rag_lora_r16', 'rag_lora_instruct']

seen_authors = set()
for (author, q), items in by_aq.items():
    if author in seen_authors:
        continue
    seen_authors.add(author)
    print(f'\n{"="*90}\nАвтор: {author}\nВопрос: {q}\n{"="*90}')
    for s in sorted(items, key=lambda x: CONDITION_ORDER.index(x['condition']) if x['condition'] in CONDITION_ORDER else 99):
        print(f'\n[{s["condition"]}]')
        print(s['answer'][:500])
    if len(seen_authors) == 3:
        break

## 7. Архив результатов

После завершения zip лежит в Output панели Kaggle — там его можно скачать.

In [ ]:
!cd models && zip -r /kaggle/working/glyph_evaluation_results.zip evaluation/
!ls -lh /kaggle/working/glyph_evaluation_results.zip
print('\nГотово — архив в Output панели Kaggle.')